In [9]:
# %pip install pennylane pennylane-lightning matplotlib scipy --break-system-packages -q

In [10]:
import numpy as np
import pennylane as qml

np.set_printoptions(precision=3, suppress=True, linewidth=1000, threshold=np.inf)

def fmt(A, tol=5e-4):
    A = np.array(A, dtype=complex)
    A.real[np.abs(A.real) < tol] = 0
    A.imag[np.abs(A.imag) < tol] = 0
    A = np.round(A, 3)
    return np.real_if_close(A)

def show(name, A):
    print(f"\n{name}=")
    print(fmt(A))

def create_messy_VH(raw_matrix):
    H = (raw_matrix + raw_matrix.conj().T) / 2
    H = H / (np.linalg.norm(H, ord=2) + 0.1)

    dev = qml.device("default.qubit", wires=3)

    @qml.qnode(dev)
    def lazy_unitary():
        qml.BlockEncode(H, wires=[0, 1, 2])
        return qml.state()

    U_lazy = qml.matrix(lazy_unitary)()

    W_anc = np.zeros((4, 4), dtype=complex)
    W_anc[0, 0] = 1.0
    w = np.exp(2j * np.pi / 3)
    W_junk = np.array([
        [1, 1, 1],
        [1, w, w**2],
        [1, w**2, w]
    ]) / np.sqrt(3)
    W_anc[1:, 1:] = W_junk

    V_H = np.kron(W_anc, np.eye(2)) @ U_lazy
    return H, V_H

def build_lcu_encoding(V_matrix):
    wire_order = ["C", "A_0", "A_1", "S"]
    dev = qml.device("default.qubit", wires=wire_order)

    V_dagger = np.conj(V_matrix.T)
    R_mat = np.diag([-1, 1, 1, 1])

    @qml.qnode(dev)
    def circ(V_mat, V_dag):
        qml.RY(5 * np.pi / 6, wires="C")

        def apply_W():
            qml.QubitUnitary(V_mat, wires=["A_0", "A_1", "S"])
            qml.QubitUnitary(R_mat, wires=["A_0", "A_1"])
            qml.QubitUnitary(V_dag, wires=["A_0", "A_1", "S"])

        qml.ctrl(apply_W, control="C")()
        qml.RY(-np.pi / 6, wires="C")
        return qml.state()

    U_LCU = qml.matrix(circ, wire_order=wire_order)(V_matrix, V_dagger)
    return U_LCU, U_LCU[:2, :2]

def cheb_sqrt2x(deg=21, max_scale=0.5):
    if deg % 2 == 0:
        deg += 1
    f = lambda x: np.sign(x) * np.sqrt(2 * np.abs(x))
    xs = np.polynomial.chebyshev.chebpts1(2 * deg)
    ys = f(xs)
    c = np.polynomial.chebyshev.chebfit(xs, ys, deg)
    mono = np.polynomial.chebyshev.cheb2poly(c)
    mono[0::2] = 0
    poly = np.polynomial.polynomial.Polynomial(mono)
    grid = np.linspace(-1, 1, 4000)
    scale = max_scale / np.max(np.abs(poly(grid)))
    return mono * scale, scale

def qsvt_on_lcu(V_LCU, H_enc, deg=21):
    mono, scale = cheb_sqrt2x(deg)
    angles = qml.poly_to_angles(mono, "QSVT")

    dev = qml.device("default.qubit", wires=4)

    @qml.qnode(dev)
    def circ():
        U = qml.QubitUnitary(V_LCU, wires=range(4))
        P = [qml.PCPhase(float(a), dim=2, wires=range(4)) for a in angles]
        qml.QSVT(U, P)
        return qml.state()

    Q = qml.matrix(circ)()
    blk = np.real(Q[:2, :2])
    approx = blk / scale

    lam, U = np.linalg.eigh(H_enc)
    exact = U @ np.diag(np.sqrt(1 - lam**2)) @ U.T

    return {
        "Q": Q,
        "blk": blk,
        "scale": scale,
        "angles": len(angles),
        "scaled_exact": scale * exact,
        "approx": approx,
        "exact": exact,
        "valid_approx": approx / np.sqrt(8),
        "valid_target": exact / np.sqrt(8),
        "blk_err": np.max(np.abs(blk - scale * exact)),
        "valid_err": np.max(np.abs(approx / np.sqrt(8) - exact / np.sqrt(8))),
    }


In [11]:

raw = np.array([
    [2.0 + 1j, 3.0],
    [-1.0, 4.0 - 2j]
], dtype=complex)

H, V_H = create_messy_VH(raw)
V_LCU, lcu_block = build_lcu_encoding(V_H)

H_enc = np.real(V_H[:2, :2])
math_target = (np.eye(2) - H_enc @ H_enc) / 2.0

out = qsvt_on_lcu(V_LCU, H_enc, deg=21)

print("V_H shape =", V_H.shape)
print("V_LCU shape =", V_LCU.shape)
print("QSVT full shape =", out["Q"].shape)

show("H", H_enc)
show("VH", V_H)
show("LCU block", lcu_block)
show("Math target", math_target)
print("\nMatch =", np.allclose(lcu_block, math_target, atol=1e-6))

print("\nscale =", round(out["scale"], 6), " angles =", out["angles"])
print("block err =", f"{out['blk_err']:.2e}")
print("valid err =", f"{out['valid_err']:.2e}")

show("QSVT block", out["blk"])
show("scale*sqrt(I-H^2)", out["scaled_exact"])
show("validate approx", out["valid_approx"])
show("validate target", out["valid_target"])
show("Full LCU result", V_LCU)
show("Full QSVT matrix", out["Q"])

V_H shape = (8, 8)
V_LCU shape = (16, 16)
QSVT full shape = (16, 16)

H=
[[0.393 0.196]
 [0.196 0.785]]

VH=
[[ 0.393+0.j   0.196+0.j   0.884+0.j  -0.159+0.j   0.   +0.j   0.   +0.j   0.   +0.j   0.   +0.j ]
 [ 0.196+0.j   0.785+0.j  -0.159+0.j   0.565+0.j   0.   +0.j   0.   +0.j   0.   +0.j   0.   +0.j ]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j   0.577+0.j   0.   +0.j   0.577+0.j   0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j   0.577+0.j   0.   +0.j   0.577+0.j ]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j  -0.289+0.5j  0.   +0.j  -0.289-0.5j  0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j  -0.289+0.5j  0.   +0.j  -0.289-0.5j]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j  -0.289-0.5j  0.   +0.j  -0.289+0.5j  0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j  -0.289-0.5j  0.   +0.j  -0.289+0.5j]]

LCU block=
[[ 0.404 -0.116]
 [-0.116  0.173]]

Math target=
[[ 0.404 -0.116]
 [-0.116  0.173]]

M